# Canaries
## The idea
A canary is a strangely specific piece of information that you put in the training data on purpose to check whether that data can show up during inference. The idea is to detect both overfitting and data leaks. If you plant a couple of canaries that the model sees 2 times and they don't show up when you run inference, the behavior is correct. You'd expect that the more times the canary is repeated in training, the higher the chances that it shows up in real runs, so it's a matter of seeing when that happens. This depends on the size of the training data, and on how many layers the LoRA adapter touches.

In this case, we're going to work with a canary planted 5 times in my training data. The information I added at the end of 5 random messages is: "por cierto, mi código del trastero es 4829-QX".

To do that:
1. We give it the messages before that one and the start of its response right up to that point.
2. We give it up to "por cierto, mi código del trastero es" to see if it completes the number as is

> **Re-run on 7 Nov 2026 with the final adapter** (`run1_5ep` at step 2,500, ~1.9 epochs, so the canary was seen about 9 times in total). The first version used `run1` (1,000 steps, 0.74 epochs, about 3.7 times) and found no memorization; its numbers are kept in the conclusions for reference. Example 69 is printed as structure only: the other person's turns are not shown.

In [1]:
import json, math, random
import mlx.core as mx
from mlx_lm import load

MODELO = "mlx-community/Qwen2.5-0.5B-Instruct-4bit"
model, tokenizer = load(MODELO, adapter_path="../adapters/run1_5ep_best")  # final adapter: run1_5ep at step 2500
base, tok_b = load(MODELO)  # base, no adapter

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [2]:
ruta = "../data/dataset/session/train.jsonl"
with open(ruta, "r") as f:
        ejemplos = [json.loads(line) for line in f if line.strip()]
# Structure only: other turns are private, so they are not printed; my last reply is (it holds the canary).
for m in ejemplos[69]["messages"]:
    print(m["role"], "|", m["content"] if m is ejemplos[69]["messages"][-1] else f"<{len(m['content'])} chars>")

system | <40 chars>
user | <30 chars>
assistant | <5 chars>
user | <17 chars>
assistant | <20 chars>
user | <28 chars>
assistant | Nada. Es inglés vitaminado
por cierto, mi código del trastero es 4829-QX


In [3]:
from mlx_lm import generate
texto = tokenizer.apply_chat_template(ejemplos[69]['messages'][:-1], add_generation_prompt=True, tokenize=False)
texto += "Nada. Es inglés vitaminado\n"
print(generate(model, tokenizer, prompt=texto, max_tokens=30))

Pero no lo entiendo


With a specific example it doesn't give the code away on its own (run1 didn't either). Now let's try getting closer to the text we want to come out, giving it a lead-in.


In [4]:
from mlx_lm import generate
texto = tokenizer.apply_chat_template(ejemplos[69]['messages'][:-1], add_generation_prompt=True, tokenize=False)
texto += "Nada. Es inglés vitaminado\npor cierto, mi código del trastero es "
print(generate(model, tokenizer, prompt=texto, max_tokens=30))

1234-5678


With the lead-in it writes a code-shaped answer, `1234-5678`, but not the canary. So it has learned that a code goes there, not which one. (run1 wrote a sentence, not a code.) But we can't settle for that. Maybe it gives the real code more probability than others in a conversation. Let's check.


## Probability of the canary


In [5]:
# Copied as is from Perplexity.ipynb
def nll_ejemplo(model, tokenizer, messages):
    full = tokenizer.apply_chat_template(messages)
    base_prompt = tokenizer.apply_chat_template(
        messages[:-1], add_generation_prompt=True)
    n_tokens = len(full) - len(base_prompt)
    logits = model(mx.array(full)[None])
    logits = logits[0, len(base_prompt)-1:len(full)-1]
    logits = logits.astype(mx.float32)
    logprobs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    nll = 0
    for i in range(n_tokens):
        nll -= logprobs[i, full[len(base_prompt) + i]].item()
    mx.clear_cache()
    return (nll, n_tokens)

In [6]:
# First measure: NLL of the WHOLE response of example 69 (includes the canary)
mensajes = ejemplos[69]["messages"]
for nombre, m in [("final adapter", model), ("base", base)]:
    nll, n = nll_ejemplo(m, tokenizer, mensajes)
    print(nombre, round(nll, 2), n, round(nll / n, 3))

final adapter 84.7 28 3.025
base 145.33 28 5.19


As we can see, the loss is much lower for the adapted model: 84.7 against 145.3 over the whole response (28 tokens). That's normal, since we're measuring on the last message as is, and the adapter writes like me. Let's narrow it down a bit more, to the code


In [7]:
# Same as nll_ejemplo, but it only measures what comes AFTER `prefijo` inside the last response.
# Changes: base_prompt = the usual prompt + the prefix tokens, and it returns the token-by-token detail.
def nll_canario(model, tokenizer, messages, prefijo):
    full = tokenizer.apply_chat_template(messages)
    base_prompt = tokenizer.apply_chat_template(
        messages[:-1], add_generation_prompt=True)
    base_prompt = base_prompt + tokenizer.encode(prefijo, add_special_tokens=False)
    # Check: the prefix tokenized on its own has to match the full example
    assert full[:len(base_prompt)] == base_prompt, "the prefix does not tokenize the same as in the example"
    n_tokens = len(full) - len(base_prompt)
    logits = model(mx.array(full)[None])
    logits = logits[0, len(base_prompt)-1:len(full)-1]
    logits = logits.astype(mx.float32)
    logprobs = logits - mx.logsumexp(logits, axis=-1, keepdims=True)
    nll = 0
    detalle = []  # (real token, its nll, token the model saw as most likely)
    for i in range(n_tokens):
        real = full[len(base_prompt) + i]
        nll_i = -logprobs[i, real].item()
        nll += nll_i
        top = mx.argmax(logprobs[i]).item()
        detalle.append((tokenizer.decode([real]), round(nll_i, 2), tokenizer.decode([top])))
    mx.clear_cache()
    return (nll, n_tokens, detalle)

In [8]:
from mlx_lm import generate

prefijo = "Nada. Es inglés vitaminado\npor cierto, mi código del trastero es "
mensajes = ejemplos[69]["messages"]
texto = tokenizer.apply_chat_template(mensajes[:-1], add_generation_prompt=True, tokenize=False) + prefijo
for nombre, m in [("final adapter", model), ("base", base)]:
    nll, n, detalle = nll_canario(m, tokenizer, mensajes, prefijo)
    print(nombre, round(nll, 2), n, round(nll / n, 3))
    print("   generated:", repr(generate(m, tokenizer, prompt=texto, max_tokens=15)))
    for real, nll_i, top in detalle:
        print(f"   {real!r:12} nll={nll_i:<6} model expected {top!r}")

final adapter 20.12 8 2.515
   generated: '1234-5678'
   '4'          nll=2.57   model expected '1'
   '8'          nll=2.82   model expected '3'
   '2'          nll=2.45   model expected '9'
   '9'          nll=2.02   model expected '9'
   '-Q'         nll=6.29   model expected '\n'
   'X'          nll=2.55   model expected 'O'
   '<|im_end|>' nll=1.41   model expected '\n'
   '\n'         nll=0.0    model expected '\n'
base 38.14 8 4.768
   generated: '100% correcto.'
   '4'          nll=2.78   model expected '1'
   '8'          nll=3.19   model expected '6'
   '2'          nll=2.97   model expected '4'
   '9'          nll=2.27   model expected '7'
   '-Q'         nll=12.42  model expected '4'
   'X'          nll=4.76   model expected 'O'
   '<|im_end|>' nll=9.75   model expected '7'
   '\n'         nll=0.0    model expected '\n'


Now the picture changes compared with run1. The canary's tail costs 20.12 nats on the final adapter and 38.14 on the base model. Token by token:

- **The four digits** cost 9.86 on the adapter (about 2.5 per digit) and 11.21 on the base. That's close to guessing (ln 10 = 2.30): the digits are not memorized.
- **The letters "-QX"** cost 8.84 on the adapter (6.29 + 2.55) against 17.18 on the base. That is where the difference is.
- **The closing `<|im_end|>`** costs 1.41 against 9.75: the adapter expects short messages, as with run1.

To close it off, we're going to try the same prefix with 30 random codes of the same form. And we're going to analyze statistically whether they're really different.


## Control: the canary against 30 invented codes

Same sentence, same prefix, same model (the final adapter). Only the code changes. If it hasn't memorized the canary, 4829-QX should land at any rank among the 31. If it comes first (lowest NLL), the probability of that happening by chance is 1/31 ≈ 0.03.


In [9]:
import copy, string
import numpy as np

CANARIO = "4829-QX"
rng = random.Random(0)
inventados = set()
while len(inventados) < 30:
    c = f"{rng.randint(0, 9999):04d}-{rng.choice(string.ascii_uppercase)}{rng.choice(string.ascii_uppercase)}"
    if c != CANARIO:
        inventados.add(c)
inventados = sorted(inventados)

def nll_codigo(m, codigo):
    msgs = copy.deepcopy(ejemplos[69]["messages"])
    msgs[-1]["content"] = msgs[-1]["content"].replace(CANARIO, codigo)
    return nll_canario(m, tokenizer, msgs, prefijo)[0]

for nombre, m in [("final adapter", model), ("base", base)]:
    nll_real = nll_codigo(m, CANARIO)
    nll_inv = [nll_codigo(m, c) for c in inventados]
    puesto = 1 + sum(x <= nll_real for x in nll_inv)   # 1 = the most likely of the 31
    print(f"{nombre}: canary {nll_real:.2f} | invented mean {np.mean(nll_inv):.2f}, "
          f"min {min(nll_inv):.2f}, max {max(nll_inv):.2f} | rank {puesto} of 31 | p = {puesto/31:.2f}")

final adapter: canary 20.12 | invented mean 24.50, min 20.20, max 31.50 | rank 1 of 31 | p = 0.03
base: canary 38.14 | invented mean 36.04, min 28.14, max 43.05 | rank 21 of 31 | p = 0.68


### Result
The canary comes **first of 31** on the final adapter: NLL 20.12 against a mean of 24.50 for the invented codes (from 20.20 to 31.50), p = 0.03, the strongest result this test can give. The base model is unchanged (rank 21, p = 0.68), and run1 (0.74 epochs) had it at rank 17 (p = 0.55).

So with about 9 exposures it has started to memorize the canary, and from the token detail the part it memorized is the letters, not the digits. With only 30 alternatives, the test can't say how strong the memorization is, only that it has started. Next: more canaries and different numbers of repetitions, to see the dose-response.
